## Analyzing Covariance Matrices of the simulations

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import sys

esmDMS_DIR = "/Users/dylanwells/popDMS/esmDMS"

sys.path.insert(0, esmDMS_DIR)
from esmdmsfunctions import (
    get_eigenvector_simulation_results,
    get_simulation_results,
    generate_selection, gaussian_selection, zero_selection,
    z_normalize, load_final_df,
)
from scipy.stats import pearsonr


In [2]:
import os

# ── Paths to analyse (name → directory containing layer0/, layer1/, … subdirs)
comb_path = esmDMS_DIR + "/data/bg_bf_comb_data"
bg_path   = esmDMS_DIR + "/data/inference_data/BG505"
bf_path   = esmDMS_DIR + "/data/inference_results"

PATHS = {
    "comb":  comb_path,
    "BG505": bg_path,
    "BF520": bf_path,
}

# ── Layers to analyze
LAYERS = range(31)

# ── Simulation parameters
N_GENS      = 30      # number of generations to simulate
SAVE_EVERY  = 1       # save counts every N generations
FITNESS_FN  = 'exp'   # 'plus1' or 'exp'

# ── Eigenvector / PCA parameters
VARIANCE_EXPLAINED_CUTOFF = 0.8   # keep PCs that together explain this fraction of variance
WEIGHT_BY_INITIAL_FREQ    = False  # weight covariance matrix by pre-selection counts

# ── Selection function (generate_selection, gaussian_selection, or zero_selection)
SEL_FUNC = gaussian_selection

# ── Inference flags
RUN_INFERENCE    = True
RUN_GAMMA        = False
CALC_ERROR_BARS  = False
INFER_IGNORED    = True

In [ ]:

# Full covariance matrix method seems to be the best

PATHS = {
    "comb":  comb_path,
    "BG505": bg_path,
    "BF520": bf_path,
}


all_results_fullcov = {}

for path_name, path in PATHS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {path_name}  |  get_simulation_results, method=fullcov")
    print('='*60)

    res = get_simulation_results(
        n_gens=N_GENS,
        embedding_df_path=path,
        sel_func=SEL_FUNC,
        inference=RUN_INFERENCE,
        gamma_analysis=RUN_GAMMA,
        fitness=FITNESS_FN,
        save_every=SAVE_EVERY,
        layers=LAYERS,
        calc_error_bars=True,
        infer_ignored_dims=INFER_IGNORED,
        method="fullcov",
    )
    all_results_fullcov[path_name] = res  
print("\nDone. Datasets completed:", list(all_results_fullcov.keys()))


Dataset: comb  |  get_simulation_results, method=fullcov
Running layer 0...
Running simulation...
  Analyzing generation 30...
Transferring simulation data to inference dataframe format...
Running inference calculations on simulated data (method='fullcov')...
Running layer 1...
Running simulation...
  Analyzing generation 30...
Transferring simulation data to inference dataframe format...
Running inference calculations on simulated data (method='fullcov')...
Running layer 2...
Running simulation...
  Analyzing generation 30...
Transferring simulation data to inference dataframe format...
Running inference calculations on simulated data (method='fullcov')...
Running layer 3...
Running simulation...
  Analyzing generation 30...
Transferring simulation data to inference dataframe format...
Running inference calculations on simulated data (method='fullcov')...
Running layer 4...
Running simulation...
  Analyzing generation 30...
Transferring simulation data to inference dataframe format..

In [ ]:
# save the fullcov data for later analysis
pwd = "/Users/dylanwells/popDMS/esmDMS"
save_dir = pwd + "/esm_sim_saves"
output_file = "fullcov_all_results_cov_mats.pkl"

output_path = os.path.join(save_dir, output_file)

with open(output_path, "wb") as f:
    pickle.dump(all_results_fullcov, f)
print(f"\nFull covariance method all results saved to {output_path}")

## Marchenko-Pastur Eigenvalue Analysis

For each layer and dataset, we extract the time-integrated covariance matrix $\Sigma$ (summed over replicates) saved by `get_simulation_results` and compare its eigenvalue spectrum to a Marchenko-Pastur (MP) null distribution.

**MP parameterisation via moment-matching** — rather than needing the explicit number of sequences $N$, we estimate the MP parameters directly from the empirical eigenvalues:

$$\sigma^2 = \frac{\text{tr}(\Sigma)}{d}, \qquad q = \frac{\text{Var}[\lambda]}{\sigma^4}$$

The MP bulk eigenvalue support is then $\bigl[\sigma^2(1-\sqrt{q})^2,\; \sigma^2(1+\sqrt{q})^2\bigr]$ and the density is

$$\rho(\lambda) = \frac{\sqrt{(\lambda_+ - \lambda)(\lambda - \lambda_-)}}{2\pi\,\sigma^2\,q\,\lambda}, \quad \lambda \in [\lambda_-, \lambda_+]$$

Eigenvalues above $\lambda_+$ are candidate signal components.

In [ ]:
def marchenko_pastur_density(lam, sigma2, q):
    """Marchenko-Pastur density evaluated at eigenvalue(s) lam.

    Parameters
    ----------
    lam    : array-like  eigenvalue(s)
    sigma2 : float       noise variance (estimated as trace(C)/d)
    q      : float       aspect ratio d/N  (estimated via moment-matching)

    Returns
    -------
    rho : ndarray  MP density (zero outside the bulk support)
    """
    lam = np.asarray(lam, dtype=float)
    lam_plus  = sigma2 * (1 + np.sqrt(q)) ** 2
    lam_minus = sigma2 * (1 - np.sqrt(q)) ** 2
    in_bulk   = (lam >= lam_minus) & (lam <= lam_plus)
    rho       = np.zeros_like(lam)
    lam_b     = lam[in_bulk]
    rho[in_bulk] = (
        np.sqrt((lam_plus - lam_b) * (lam_b - lam_minus))
        / (2 * np.pi * sigma2 * q * lam_b)
    )
    return rho, lam_minus, lam_plus


def mp_params_from_eigenvalues(eigvals):
    """Estimate MP sigma^2 and q from empirical eigenvalues via moment-matching.

    moment 1: E[lambda]   = sigma^2
    moment 2: E[lambda^2] = sigma^4 * (1 + q)  =>  q = E[lambda^2]/sigma^4 - 1
    """
    sigma2 = np.mean(eigvals)
    # Use second moment: E[lambda^2] - (E[lambda])^2 = sigma^4 * q  => q = Var/sigma^4
    q = np.var(eigvals) / sigma2 ** 2
    q = max(q, 1e-4)   # numerical floor
    return sigma2, q


print("Helper functions defined.")

In [ ]:
# ── Extract eigenvalues and gamma for all datasets and layers ──────────────────
# Expects all_results_fullcov[dataset_name] = (all_layer_fits,
#                                              all_sel_coeffs,
#                                              detailed_selection_results,
#                                              all_gamma_analysis,
#                                              all_generation_counts)
# detailed_selection_results[layer] = [s, s_joint, s_err, s_joint_err,
#                                      icov_sum (d×d), gamma_opt]

eig_data = {}   # eig_data[dataset][layer] = {'eigvals': ..., 'gamma_opt': ..., 'd': ...}

for dataset, res in all_results_fullcov.items():
    detailed = res[2]   # detailed_selection_results dict
    eig_data[dataset] = {}
    for layer, layer_res in detailed.items():
        if len(layer_res) < 6:
            print(f"  [{dataset}] layer {layer}: no icov_sum saved (re-run get_simulation_results)")
            continue
        icov_sum  = layer_res[4]   # (d, d) summed covariance
        gamma_opt = layer_res[5]   # float

        eigvals = np.linalg.eigvalsh(icov_sum)   # sorted ascending, real (symmetric)
        eigvals = eigvals[::-1]                   # descending order

        eig_data[dataset][layer] = {
            'eigvals':   eigvals,
            'gamma_opt': gamma_opt,
            'd':         icov_sum.shape[0],
        }

# Quick summary
for dataset in eig_data:
    layers_ok = list(eig_data[dataset].keys())
    print(f"{dataset}: {len(layers_ok)} layers with eigenvalue data")

In [ ]:
# ── Per-dataset: eigenvalue spectrum vs MP for every layer ────────────────────
# One figure per dataset; subplots arranged as grid over layers.

N_COLS = 4   # columns in the subplot grid

for dataset, layer_dict in eig_data.items():
    layers = sorted(layer_dict.keys())
    if not layers:
        continue

    n_layers = len(layers)
    n_rows   = int(np.ceil(n_layers / N_COLS))
    fig, axes = plt.subplots(n_rows, N_COLS,
                             figsize=(4 * N_COLS, 3 * n_rows),
                             constrained_layout=True)
    axes = np.array(axes).flatten()

    for ax_idx, layer in enumerate(layers):
        ax   = axes[ax_idx]
        info = layer_dict[layer]
        ev   = info['eigvals']
        d    = info['d']
        gopt = info['gamma_opt']

        # Keep only positive eigenvalues (icov can have small negatives from numerics)
        ev_pos = ev[ev > 0]

        # Fit MP parameters by moment-matching
        sigma2, q = mp_params_from_eigenvalues(ev_pos)
        _, lam_min, lam_max = marchenko_pastur_density(np.array([sigma2]), sigma2, q)

        # --- histogram of empirical eigenvalues ---
        ax.hist(ev_pos, bins=40, density=True, color='steelblue', alpha=0.6,
                label='empirical')

        # --- MP density curve ---
        lam_grid = np.linspace(max(lam_min * 0.5, 1e-12), lam_max * 1.5, 400)
        rho, _, _ = marchenko_pastur_density(lam_grid, sigma2, q)
        ax.plot(lam_grid, rho, color='tomato', lw=2, label='MP fit')

        # --- mark MP bulk edge and gamma_opt ---
        ax.axvline(lam_max,  color='tomato',   lw=1.2, ls='--', label=r'$\lambda_+$')
        ax.axvline(gopt,     color='goldenrod', lw=1.2, ls=':',  label=r'$\gamma_\mathrm{opt}$')

        n_signal = int(np.sum(ev_pos > lam_max))
        ax.set_title(f"layer {layer}  |  {n_signal}/{d} signal dims\n"
                     f"q={q:.2f}  γ={gopt:.3g}", fontsize=8)
        ax.set_xlabel(r'eigenvalue $\lambda$', fontsize=8)
        ax.set_ylabel('density', fontsize=8)
        ax.tick_params(labelsize=7)
        if ax_idx == 0:
            ax.legend(fontsize=6, loc='upper right')

    # hide unused axes
    for ax in axes[n_layers:]:
        ax.set_visible(False)

    fig.suptitle(f"{dataset} — eigenvalue spectra vs Marchenko-Pastur", fontsize=13)
    plt.show()

In [ ]:
# ── Summary: number of signal dimensions above MP bulk edge per layer ──────────
# One line per dataset, x-axis = layer, y-axis = n_signal / d

fig, ax = plt.subplots(figsize=(10, 4))

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
for ds_idx, (dataset, layer_dict) in enumerate(eig_data.items()):
    layers = sorted(layer_dict.keys())
    signal_fracs = []
    for layer in layers:
        info  = layer_dict[layer]
        ev    = info['eigvals']
        d     = info['d']
        ev_pos = ev[ev > 0]
        sigma2, q = mp_params_from_eigenvalues(ev_pos)
        lam_max   = sigma2 * (1 + np.sqrt(q)) ** 2
        signal_fracs.append(np.sum(ev_pos > lam_max) / d)
    ax.plot(layers, signal_fracs, marker='o', label=dataset, color=colors[ds_idx % len(colors)])

ax.set_xlabel('ESM-2 layer')
ax.set_ylabel('fraction of dims above MP bulk edge')
ax.set_title('Signal fraction across layers (eigenvalues > $\\lambda_+$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()